# Assignment 6: Machine Translation System
## English ↔ Hindi Translation for Public Information Content

This notebook implements a Machine Translation system to translate public information content between English and Hindi (an Indian language).

### Topics Covered:
1. Introduction to Neural Machine Translation
2. Using Pre-trained Translation Models (Hugging Face Transformers)
3. English to Hindi Translation
4. Hindi to English Translation
5. Translating Public Information Content
6. Evaluation using BLEU Score

## 1. Install Required Libraries

In [1]:
# Install required packages
!pip install transformers torch sentencepiece sacremoses sacrebleu

  Using cached pywin32-311-cp310-cp310-win_amd64.whl.metadata (10 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.4 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/4.0 MB 2.4 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.0 MB 2.2 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.0 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.6 MB/s  0:00:01
Using cached pywin32-311-cp310-cp310-win_amd64.whl (9.6 MB)

   ---------------------------------------- 0/5 [pywin32]
   ---------------------------------------- 0/5 [pywin32]
   ---------------------------------------- 0/5 [pywin32]
   ---------------------------------------- 0/5 [pywin32]
   ---------------------------------------- 0/5 [pywin32]
   ---------------------------------------- 0/5 [pywin32]
   ---


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Libraries

In [2]:
import warnings
warnings.filterwarnings('ignore')

from transformers import MarianMTModel, MarianTokenizer
from transformers import pipeline
import torch
import sacrebleu

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Libraries imported successfully!
PyTorch version: 2.7.1+cu118
CUDA available: True


## 3. Load Translation Models

We use Helsinki-NLP's MarianMT models from Hugging Face:
- **English → Hindi**: `Helsinki-NLP/opus-mt-en-hi`
- **Hindi → English**: `Helsinki-NLP/opus-mt-hi-en`

In [3]:
# English to Hindi Model
print("Loading English to Hindi translation model...")
en_hi_model_name = "Helsinki-NLP/opus-mt-en-hi"
en_hi_tokenizer = MarianTokenizer.from_pretrained(en_hi_model_name)
en_hi_model = MarianMTModel.from_pretrained(en_hi_model_name)
print("English to Hindi model loaded successfully!")

Loading English to Hindi translation model...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

English to Hindi model loaded successfully!


In [4]:
# Hindi to English Model
print("Loading Hindi to English translation model...")
hi_en_model_name = "Helsinki-NLP/opus-mt-hi-en"
hi_en_tokenizer = MarianTokenizer.from_pretrained(hi_en_model_name)
hi_en_model = MarianMTModel.from_pretrained(hi_en_model_name)
print("Hindi to English model loaded successfully!")

Loading Hindi to English translation model...


model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

Hindi to English model loaded successfully!


## 4. Define Translation Functions

In [5]:
def translate_en_to_hi(text):
    """
    Translate English text to Hindi
    
    Args:
        text (str): English text to translate
    
    Returns:
        str: Translated Hindi text
    """
    # Tokenize the input text
    inputs = en_hi_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    
    # Generate translation
    with torch.no_grad():
        translated = en_hi_model.generate(**inputs)
    
    # Decode the translated text
    translated_text = en_hi_tokenizer.decode(translated[0], skip_special_tokens=True)
    
    return translated_text


def translate_hi_to_en(text):
    """
    Translate Hindi text to English
    
    Args:
        text (str): Hindi text to translate
    
    Returns:
        str: Translated English text
    """
    # Tokenize the input text
    inputs = hi_en_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    
    # Generate translation
    with torch.no_grad():
        translated = hi_en_model.generate(**inputs)
    
    # Decode the translated text
    translated_text = hi_en_tokenizer.decode(translated[0], skip_special_tokens=True)
    
    return translated_text


print("Translation functions defined successfully!")

Translation functions defined successfully!


## 5. Test Basic Translation

In [6]:
# Test English to Hindi Translation
test_sentences_en = [
    "Hello, how are you?",
    "Welcome to India.",
    "Education is the key to success.",
    "Please wash your hands regularly.",
    "The weather is very pleasant today."
]

print("=" * 70)
print("ENGLISH TO HINDI TRANSLATION")
print("=" * 70)

for sentence in test_sentences_en:
    hindi_translation = translate_en_to_hi(sentence)
    print(f"\nEnglish: {sentence}")
    print(f"Hindi:   {hindi_translation}")
    print("-" * 50)

ENGLISH TO HINDI TRANSLATION

English: Hello, how are you?
Hindi:   हैलो, तुम कैसे हो?
--------------------------------------------------

English: Welcome to India.
Hindi:   भारत में आपका स्वागत है।
--------------------------------------------------

English: Education is the key to success.
Hindi:   शिक्षा सफलता की कुंजी है ।
--------------------------------------------------

English: Please wash your hands regularly.
Hindi:   कृपया नियमित रूप से अपने हाथों को धोइए ।
--------------------------------------------------

English: The weather is very pleasant today.
Hindi:   मौसम आज बहुत सुखद है.
--------------------------------------------------


In [7]:
# Test Hindi to English Translation
test_sentences_hi = [
    "नमस्ते, आप कैसे हैं?",
    "भारत में आपका स्वागत है।",
    "शिक्षा सफलता की कुंजी है।",
    "कृपया अपने हाथ नियमित रूप से धोएं।",
    "आज मौसम बहुत सुहावना है।"
]

print("=" * 70)
print("HINDI TO ENGLISH TRANSLATION")
print("=" * 70)

for sentence in test_sentences_hi:
    english_translation = translate_hi_to_en(sentence)
    print(f"\nHindi:   {sentence}")
    print(f"English: {english_translation}")
    print("-" * 50)

HINDI TO ENGLISH TRANSLATION

Hindi:   नमस्ते, आप कैसे हैं?
English: Hello, how are you?
--------------------------------------------------

Hindi:   भारत में आपका स्वागत है।
English: Welcome to India.
--------------------------------------------------

Hindi:   शिक्षा सफलता की कुंजी है।
English: Education is the key to success.
--------------------------------------------------

Hindi:   कृपया अपने हाथ नियमित रूप से धोएं।
English: Please wash your hands regularly.
--------------------------------------------------

Hindi:   आज मौसम बहुत सुहावना है।
English: The weather is very beautiful today.
--------------------------------------------------


## 6. Translating Public Information Content

Now let's translate various types of public information content including:
- Government announcements
- Health advisories
- Educational information
- Public safety messages
- Transportation information

In [8]:
# Public Information Content - English
public_info_english = {
    "Government Notice": [
        "All citizens must carry valid identification documents.",
        "The new tax policy will be effective from next month.",
        "Voter registration deadline is approaching. Register now."
    ],
    "Health Advisory": [
        "Drink plenty of water and stay hydrated during summer.",
        "Vaccination is important for preventing diseases.",
        "Consult a doctor if you experience any symptoms."
    ],
    "Public Safety": [
        "Always wear a helmet while riding a motorcycle.",
        "Do not drink and drive. Stay safe on roads.",
        "In case of emergency, call the helpline number."
    ],
    "Education": [
        "School admissions are now open for the academic year.",
        "Free textbooks will be distributed to all students.",
        "Online classes are available for remote learners."
    ],
    "Transportation": [
        "The metro service will be available from 6 AM to 11 PM.",
        "Bus fares have been revised. Please check the new rates.",
        "Traffic diversions are in place due to road construction."
    ]
}

print("=" * 80)
print("PUBLIC INFORMATION CONTENT: ENGLISH TO HINDI TRANSLATION")
print("=" * 80)

for category, messages in public_info_english.items():
    print(f"\n{'='*40}")
    print(f"Category: {category}")
    print(f"{'='*40}")
    
    for msg in messages:
        hindi_translation = translate_en_to_hi(msg)
        print(f"\n📌 English: {msg}")
        print(f"📌 Hindi:   {hindi_translation}")

PUBLIC INFORMATION CONTENT: ENGLISH TO HINDI TRANSLATION

Category: Government Notice

📌 English: All citizens must carry valid identification documents.
📌 Hindi:   सभी नागरिकों को वैध पहचान दस्तावेज़ों को उठाना चाहिए.

📌 English: The new tax policy will be effective from next month.
📌 Hindi:   नया कर नीति अगले महीने से प्रभावी होगी.

📌 English: Voter registration deadline is approaching. Register now.
📌 Hindi:   वोर जोर मर्लाइन के पास आ रहा है. अब पंजीकृत करें.

Category: Health Advisory

📌 English: Drink plenty of water and stay hydrated during summer.
📌 Hindi:   गर्मियों के दौरान भरपूर पानी पी लीजिए ।

📌 English: Vaccination is important for preventing diseases.
📌 Hindi:   बीमारी को रोकने के लिए खून - खराबा ज़रूरी है ।

📌 English: Consult a doctor if you experience any symptoms.
📌 Hindi:   अगर आपको कोई लक्षण नज़र आए, तो डॉक्टर से सलाह लीजिए ।

Category: Public Safety

📌 English: Always wear a helmet while riding a motorcycle.
📌 Hindi:   हमेशा एक साइकिल चलाते समय टोप पहने रहें.

📌 En

In [9]:
# Public Information Content - Hindi
public_info_hindi = {
    "सरकारी सूचना": [
        "सभी नागरिकों को वैध पहचान पत्र रखना अनिवार्य है।",
        "नई कर नीति अगले महीने से लागू होगी।",
        "मतदाता पंजीकरण की अंतिम तिथि निकट है। अभी पंजीकरण करें।"
    ],
    "स्वास्थ्य सलाह": [
        "गर्मियों में खूब पानी पिएं और हाइड्रेटेड रहें।",
        "बीमारियों से बचाव के लिए टीकाकरण महत्वपूर्ण है।",
        "यदि आपको कोई लक्षण दिखे तो डॉक्टर से परामर्श करें।"
    ],
    "सार्वजनिक सुरक्षा": [
        "मोटरसाइकिल चलाते समय हमेशा हेलमेट पहनें।",
        "शराब पीकर गाड़ी न चलाएं। सड़कों पर सुरक्षित रहें।",
        "आपातकाल में हेल्पलाइन नंबर पर कॉल करें।"
    ]
}

print("=" * 80)
print("PUBLIC INFORMATION CONTENT: HINDI TO ENGLISH TRANSLATION")
print("=" * 80)

for category, messages in public_info_hindi.items():
    print(f"\n{'='*40}")
    print(f"Category: {category}")
    print(f"{'='*40}")
    
    for msg in messages:
        english_translation = translate_hi_to_en(msg)
        print(f"\n📌 Hindi:   {msg}")
        print(f"📌 English: {english_translation}")

PUBLIC INFORMATION CONTENT: HINDI TO ENGLISH TRANSLATION

Category: सरकारी सूचना

📌 Hindi:   सभी नागरिकों को वैध पहचान पत्र रखना अनिवार्य है।
📌 English: All citizens have a valid identity.

📌 Hindi:   नई कर नीति अगले महीने से लागू होगी।
📌 English: New policy will apply from the next month.

📌 Hindi:   मतदाता पंजीकरण की अंतिम तिथि निकट है। अभी पंजीकरण करें।
📌 English: The final date of the voted registration is at hand, but now register.

Category: स्वास्थ्य सलाह

📌 Hindi:   गर्मियों में खूब पानी पिएं और हाइड्रेटेड रहें।
📌 English: Stay a lot of water drinks and Hydroids in the summer.

📌 Hindi:   बीमारियों से बचाव के लिए टीकाकरण महत्वपूर्ण है।
📌 English: It's important for prevention of disease.

📌 Hindi:   यदि आपको कोई लक्षण दिखे तो डॉक्टर से परामर्श करें।
📌 English: If you see a sign, consult the doctor.

Category: सार्वजनिक सुरक्षा

📌 Hindi:   मोटरसाइकिल चलाते समय हमेशा हेलमेट पहनें।
📌 English: Always wear a helmet while driving.

📌 Hindi:   शराब पीकर गाड़ी न चलाएं। सड़कों पर सुरक्ष

## 7. Batch Translation Function

In [10]:
def batch_translate(texts, direction="en_to_hi"):
    """
    Translate a batch of texts
    
    Args:
        texts (list): List of texts to translate
        direction (str): Translation direction - 'en_to_hi' or 'hi_to_en'
    
    Returns:
        list: List of translated texts
    """
    if direction == "en_to_hi":
        tokenizer = en_hi_tokenizer
        model = en_hi_model
    else:
        tokenizer = hi_en_tokenizer
        model = hi_en_model
    
    # Tokenize all texts
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    
    # Generate translations
    with torch.no_grad():
        translated = model.generate(**inputs)
    
    # Decode all translations
    translated_texts = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
    
    return translated_texts


# Test batch translation
batch_texts = [
    "Government offices will remain closed on public holidays.",
    "Please stand in queue and maintain social distancing.",
    "Free medical camps will be organized next week."
]

print("Batch Translation Test:")
print("=" * 60)

hindi_translations = batch_translate(batch_texts, "en_to_hi")

for eng, hin in zip(batch_texts, hindi_translations):
    print(f"\nEnglish: {eng}")
    print(f"Hindi:   {hin}")

Batch Translation Test:

English: Government offices will remain closed on public holidays.
Hindi:   सरकारी कार्यालय सार्वजनिक छुट्टियों में बंद हो जाएगा।

English: Please stand in queue and maintain social distancing.
Hindi:   कृपया कतार में खड़े रहें और सामाजिक पतन को बनाए रखें.

English: Free medical camps will be organized next week.
Hindi:   अगले सप्ताह मुक्‍त चिकित्सीय शिविरों को संगठित किया जाएगा ।


## 8. Evaluation Using BLEU Score

BLEU (Bilingual Evaluation Understudy) Score is a metric for evaluating machine translation quality by comparing the machine translation output with reference translations.

In [11]:
def calculate_bleu_score(hypothesis, reference):
    """
    Calculate BLEU score for a single translation
    
    Args:
        hypothesis (str): Machine translated text
        reference (str): Reference (human) translation
    
    Returns:
        float: BLEU score
    """
    bleu = sacrebleu.sentence_bleu(hypothesis, [reference])
    return bleu.score


def calculate_corpus_bleu(hypotheses, references):
    """
    Calculate BLEU score for a corpus of translations
    
    Args:
        hypotheses (list): List of machine translated texts
        references (list): List of reference translations
    
    Returns:
        float: Corpus BLEU score
    """
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score


print("BLEU score functions defined!")

BLEU score functions defined!


In [12]:
# Evaluation with reference translations
# Test set with English sentences and their Hindi references

test_pairs = [
    {
        "english": "Water is essential for life.",
        "hindi_reference": "जल जीवन के लिए आवश्यक है।"
    },
    {
        "english": "Education empowers people.",
        "hindi_reference": "शिक्षा लोगों को सशक्त बनाती है।"
    },
    {
        "english": "Health is wealth.",
        "hindi_reference": "स्वास्थ्य ही धन है।"
    },
    {
        "english": "India is a democratic country.",
        "hindi_reference": "भारत एक लोकतांत्रिक देश है।"
    },
    {
        "english": "Please follow traffic rules.",
        "hindi_reference": "कृपया यातायात नियमों का पालन करें।"
    }
]

print("=" * 80)
print("TRANSLATION EVALUATION WITH BLEU SCORES")
print("=" * 80)

hypotheses = []
references = []

for pair in test_pairs:
    # Get machine translation
    machine_translation = translate_en_to_hi(pair["english"])
    
    # Calculate BLEU score
    bleu_score = calculate_bleu_score(machine_translation, pair["hindi_reference"])
    
    hypotheses.append(machine_translation)
    references.append(pair["hindi_reference"])
    
    print(f"\n📝 English:           {pair['english']}")
    print(f"🤖 Machine Translation: {machine_translation}")
    print(f"✅ Reference:          {pair['hindi_reference']}")
    print(f"📊 BLEU Score:         {bleu_score:.2f}")
    print("-" * 60)

# Calculate corpus BLEU
corpus_bleu = calculate_corpus_bleu(hypotheses, references)
print(f"\n{'='*60}")
print(f"📈 CORPUS BLEU SCORE: {corpus_bleu:.2f}")
print(f"{'='*60}")

TRANSLATION EVALUATION WITH BLEU SCORES

📝 English:           Water is essential for life.
🤖 Machine Translation: जीवन के लिए पानी ज़रूरी है ।
✅ Reference:          जल जीवन के लिए आवश्यक है।
📊 BLEU Score:         24.45
------------------------------------------------------------

📝 English:           Education empowers people.
🤖 Machine Translation: शिक्षा लोगों को शक्‍ति देती है ।
✅ Reference:          शिक्षा लोगों को सशक्त बनाती है।
📊 BLEU Score:         24.45
------------------------------------------------------------

📝 English:           Health is wealth.
🤖 Machine Translation: स्वास्थ्य धन है.
✅ Reference:          स्वास्थ्य ही धन है।
📊 BLEU Score:         19.00
------------------------------------------------------------

📝 English:           India is a democratic country.
🤖 Machine Translation: भारत लोकतांत्रिक देश है.
✅ Reference:          भारत एक लोकतांत्रिक देश है।
📊 BLEU Score:         23.64
------------------------------------------------------------

📝 English:          

## 9. Interactive Translation System

In [13]:
class MachineTranslationSystem:
    """
    A complete Machine Translation System for English-Hindi translation
    """
    
    def __init__(self):
        self.en_hi_tokenizer = en_hi_tokenizer
        self.en_hi_model = en_hi_model
        self.hi_en_tokenizer = hi_en_tokenizer
        self.hi_en_model = hi_en_model
        print("Machine Translation System initialized!")
    
    def translate(self, text, source_lang="en", target_lang="hi"):
        """
        Translate text between English and Hindi
        
        Args:
            text (str): Text to translate
            source_lang (str): Source language ('en' or 'hi')
            target_lang (str): Target language ('en' or 'hi')
        
        Returns:
            str: Translated text
        """
        if source_lang == "en" and target_lang == "hi":
            return translate_en_to_hi(text)
        elif source_lang == "hi" and target_lang == "en":
            return translate_hi_to_en(text)
        else:
            return "Invalid language pair. Supported: en↔hi"
    
    def translate_document(self, sentences, source_lang="en", target_lang="hi"):
        """
        Translate a list of sentences
        
        Args:
            sentences (list): List of sentences to translate
            source_lang (str): Source language
            target_lang (str): Target language
        
        Returns:
            list: List of translated sentences
        """
        return [self.translate(s, source_lang, target_lang) for s in sentences]
    
    def round_trip_translation(self, text, start_lang="en"):
        """
        Perform round-trip translation (translate and back-translate)
        
        Args:
            text (str): Original text
            start_lang (str): Starting language
        
        Returns:
            dict: Original, intermediate, and back-translated text
        """
        if start_lang == "en":
            intermediate = self.translate(text, "en", "hi")
            back_translated = self.translate(intermediate, "hi", "en")
        else:
            intermediate = self.translate(text, "hi", "en")
            back_translated = self.translate(intermediate, "en", "hi")
        
        return {
            "original": text,
            "intermediate": intermediate,
            "back_translated": back_translated
        }


# Create instance
mt_system = MachineTranslationSystem()

Machine Translation System initialized!


In [14]:
# Test the Translation System
print("=" * 70)
print("MACHINE TRANSLATION SYSTEM DEMO")
print("=" * 70)

# Single translation
test_text = "The government has launched a new scheme for farmers."
print(f"\n🔹 Single Translation:")
print(f"   Input (English):  {test_text}")
print(f"   Output (Hindi):   {mt_system.translate(test_text, 'en', 'hi')}")

# Document translation
document = [
    "The new policy aims to improve public health.",
    "Citizens are encouraged to participate in community programs.",
    "Digital services are now available online."
]

print(f"\n🔹 Document Translation:")
translations = mt_system.translate_document(document)
for orig, trans in zip(document, translations):
    print(f"   EN: {orig}")
    print(f"   HI: {trans}")
    print()

MACHINE TRANSLATION SYSTEM DEMO

🔹 Single Translation:
   Input (English):  The government has launched a new scheme for farmers.
   Output (Hindi):   सरकार किसानों के लिए एक नई योजना शुरू की है.

🔹 Document Translation:
   EN: The new policy aims to improve public health.
   HI: नयी नीति - व्यवस्था जन स्वास्थ्य को बेहतर बनाने का लक्ष्य रखती है ।

   EN: Citizens are encouraged to participate in community programs.
   HI: नागरिकों को समुदाय कार्यक्रमों में भाग लेने के लिए प्रोत्साहित किया जाता है ।

   EN: Digital services are now available online.
   HI: डिजिटल सेवाएँ अब ऑनलाइन हैं.



In [15]:
# Round-trip translation test
print("=" * 70)
print("ROUND-TRIP TRANSLATION TEST")
print("=" * 70)

test_sentences = [
    "Clean drinking water is a basic human right.",
    "Every child deserves quality education.",
    "Road safety saves lives."
]

for sentence in test_sentences:
    result = mt_system.round_trip_translation(sentence, "en")
    print(f"\n📌 Original (EN):        {result['original']}")
    print(f"📌 Translated (HI):      {result['intermediate']}")
    print(f"📌 Back-translated (EN): {result['back_translated']}")
    print("-" * 60)

ROUND-TRIP TRANSLATION TEST

📌 Original (EN):        Clean drinking water is a basic human right.
📌 Translated (HI):      पीने का पानी एक मूल मानव अधिकार है ।
📌 Back-translated (EN): The drinking water is an original human authority.
------------------------------------------------------------

📌 Original (EN):        Every child deserves quality education.
📌 Translated (HI):      हर बच्चा उत्कृष्ट शिक्षा के योग्य है ।
📌 Back-translated (EN): Each child deserves an outstanding education.
------------------------------------------------------------

📌 Original (EN):        Road safety saves lives.
📌 Translated (HI):      सड़क सुरक्षा जीवन बचाता है.
📌 Back-translated (EN): Road security saves lives.
------------------------------------------------------------


## 10. Summary and Conclusion

### What We Accomplished:

1. **Built a Machine Translation System** for English ↔ Hindi translation using pre-trained transformer models.

2. **Implemented Translation Functions** for both directions:
   - English → Hindi
   - Hindi → English

3. **Translated Public Information Content** including:
   - Government notices
   - Health advisories
   - Public safety messages
   - Educational information
   - Transportation updates

4. **Evaluated Translation Quality** using BLEU scores

5. **Created a Complete Translation System Class** with features like:
   - Single text translation
   - Document translation
   - Round-trip translation

### Key Technologies Used:
- **Hugging Face Transformers** - For pre-trained translation models
- **MarianMT** - Helsinki-NLP's neural machine translation models
- **SacreBLEU** - For translation evaluation
- **PyTorch** - Deep learning framework

### Limitations:
- Translation quality depends on the training data
- Some idiomatic expressions may not translate perfectly
- Technical or domain-specific terms may need fine-tuning

### Future Improvements:
- Fine-tune models on domain-specific data
- Add support for more Indian languages
- Implement post-editing suggestions
- Create a web interface for easy access